In [1]:
import os
os.chdir("..")

In [2]:
%pwd

'd:\\Projects\\RAG_Systems\\ProductDemand\\product-demand-forecasting-system'

In [13]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    trained_model_file_path: Path
    results_file_path: Path
    metadata_file_path: Path

    train_scaled_path: Path
    test_scaled_path: Path

    train_unscaled_path: Path
    test_unscaled_path: Path

    target_column: str


In [14]:
import sys
from pathlib import Path
from src.productdemand.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.productdemand.utils.common import read_yaml, create_directories
from src.productdemand.exception.custom_exception import CustomException

class ConfigurationManager:
    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE_PATH,
            schema_filepath=SCHEMA_FILE_PATH,
        ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([Path(self.config.artifacts_root)])

   

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        schema = self.schema

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            trained_model_file_path=Path(config.trained_model_file_path),
            results_file_path=Path(config.results_file_path),
            metadata_file_path=Path(config.metadata_file_path),

            train_scaled_path=Path(config.train_scaled_path),
            test_scaled_path=Path(config.test_scaled_path),

            train_unscaled_path=Path(config.train_unscaled_path),
            test_unscaled_path=Path(config.test_unscaled_path),

            target_column=schema.TARGET_COLUMN.name
        )

        return model_trainer_config




In [19]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from xgboost import XGBRegressor

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


from src.productdemand.exception.custom_exception import CustomException
from src.productdemand.logger.custom_logger import get_logger
from src.productdemand.utils.common import *

logger = get_logger(__name__)

class ModelTrainer:
    def __init__(self, config:ModelTrainerConfig):
        self.config = config

        self.models = {
            "LinearRegression": LinearRegression(),
            "RandomForestRegressor": RandomForestRegressor(
                random_state=42, n_jobs=2
            ),
            "XGBRegressor": XGBRegressor(
                random_state=42, verbosity=0, n_jobs=2
            ),
            "AdaBoostRegressor": AdaBoostRegressor(
                random_state=42
            )
        }

        # light param grid for laptop
        self.params = {
            "LinearRegression": {
                "fit_intercept": [True, False]
            },
            "RandomForestRegressor": {
                "n_estimators": [100],
                "max_depth": [10, None],
                "min_samples_split": [2, 5]
            },
            "XGBRegressor": {
                "n_estimators": [100],
                "max_depth": [3, 5],
                "learning_rate": [0.05, 0.1]
            },
            "AdaBoostRegressor": {
                "n_estimators": [50, 100],
                "learning_rate": [0.05, 0.1]
            }
        }

    def split_features_target(self, df: pd.DataFrame):
        X = df.drop(columns=[self.config["target_column"]], axis=1)
        y = df[self.config["target_column"]]
        return X, y

    def load_data(self):
        train_scaled = pd.read_csv(self.config["train_scaled_path"])
        test_scaled = pd.read_csv(self.config["test_scaled_path"])

        train_unscaled = pd.read_csv(self.config["train_unscaled_path"])
        test_unscaled = pd.read_csv(self.config["test_unscaled_path"])

        X_train_scaled, y_train = self.split_features_target(train_scaled)
        X_test_scaled, y_test = self.split_features_target(test_scaled)

        X_train_unscaled, _ = self.split_features_target(train_unscaled)
        X_test_unscaled, _ = self.split_features_target(test_unscaled)

        return (
            X_train_scaled,
            X_test_scaled,
            y_train,
            y_test,
            X_train_unscaled,
            X_test_unscaled
        )

    def get_data_for_model(
        self,
        model_name,
        X_train_scaled,
        X_test_scaled,
        X_train_unscaled,
        X_test_unscaled
    ):
        if model_name == "LinearRegression":
            return X_train_scaled, X_test_scaled
        return X_train_unscaled, X_test_unscaled

    def evaluate_predictions(self, y_true, y_pred):
        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        return {
            "r2_score": float(r2),
            "mae": float(mae),
            "rmse": float(rmse)
        }

    def save_json(self, path, data):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w") as f:
            json.dump(data, f, indent=4)

    def initiate_model_trainer(self):
        (
            X_train_scaled,
            X_test_scaled,
            y_train,
            y_test,
            X_train_unscaled,
            X_test_unscaled
        ) = self.load_data()

        results = {}
        best_model = None
        best_model_name = None
        best_score = -np.inf

        for model_name, model in self.models.items():
            print(f"\nTraining {model_name} ...")

            X_train_current, X_test_current = self.get_data_for_model(
                model_name,
                X_train_scaled,
                X_test_scaled,
                X_train_unscaled,
                X_test_unscaled
            )

            grid_search = GridSearchCV(
                estimator=model,
                param_grid=self.params[model_name],
                cv=3,
                scoring="r2",
                n_jobs=-1,
                verbose=1
            )

            grid_search.fit(X_train_current, y_train)

            best_estimator = grid_search.best_estimator_
            y_pred = best_estimator.predict(X_test_current)

            metrics = self.evaluate_predictions(y_test, y_pred)

            results[model_name] = {
                "best_params": grid_search.best_params_,
                "r2_score": metrics["r2_score"],
                "mae": metrics["mae"],
                "rmse": metrics["rmse"],
                "input_type": "scaled" if model_name == "LinearRegression" else "unscaled"
            }

            if metrics["r2_score"] > best_score:
                best_score = metrics["r2_score"]
                best_model = best_estimator
                best_model_name = model_name

        # save best model
        Path(self.config["trained_model_file_path"]).parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(best_model, self.config["trained_model_file_path"])

        results["best_model"] = {
            "model_name": best_model_name,
            "best_r2_score": float(best_score),
            "input_type": "scaled" if best_model_name == "LinearRegression" else "unscaled"
        }

        metadata = {
            "model_name": best_model_name,
            "target_column": self.config["target_column"],
            "model_file_path": self.config["trained_model_file_path"],
            "input_type": "scaled" if best_model_name == "LinearRegression" else "unscaled"
        }

        self.save_json(self.config["results_file_path"], results)
        self.save_json(self.config["metadata_file_path"], metadata)

        return results


In [20]:
model_trainer_config = {
    "train_scaled_path": "artifacts/data_transformation/regression/train.csv",
    "test_scaled_path": "artifacts/data_transformation/regression/test.csv",
    "train_unscaled_path": "artifacts/data_transformation/ensemble/train.csv",
    "test_unscaled_path": "artifacts/data_transformation/ensemble/test.csv",

    "trained_model_file_path": "artifacts/model_trainer/best_model.joblib",
    "results_file_path": "artifacts/model_trainer/model_results.json",
    "metadata_file_path": "artifacts/model_trainer/model_metadata.json",

    "target_column": "Order_Demand"
}


In [21]:
trainer = ModelTrainer(config=model_trainer_config)
results = trainer.initiate_model_trainer()
results



Training LinearRegression ...
Fitting 3 folds for each of 2 candidates, totalling 6 fits

Training RandomForestRegressor ...
Fitting 3 folds for each of 4 candidates, totalling 12 fits

Training XGBRegressor ...
Fitting 3 folds for each of 4 candidates, totalling 12 fits

Training AdaBoostRegressor ...
Fitting 3 folds for each of 4 candidates, totalling 12 fits


{'LinearRegression': {'best_params': {'fit_intercept': True},
  'r2_score': 0.020881866981817043,
  'mae': 7334.662075245005,
  'rmse': 31364.270287642365,
  'input_type': 'scaled'},
 'RandomForestRegressor': {'best_params': {'max_depth': None,
   'min_samples_split': 5,
   'n_estimators': 100},
  'r2_score': 0.4431143111886021,
  'mae': 4607.955075816841,
  'rmse': 23653.786341213607,
  'input_type': 'unscaled'},
 'XGBRegressor': {'best_params': {'learning_rate': 0.1,
   'max_depth': 5,
   'n_estimators': 100},
  'r2_score': 0.3910089135169983,
  'mae': 5341.0107421875,
  'rmse': 24735.63777225079,
  'input_type': 'unscaled'},
 'AdaBoostRegressor': {'best_params': {'learning_rate': 0.05,
   'n_estimators': 50},
  'r2_score': 0.03217296116894408,
  'mae': 8813.762495927063,
  'rmse': 31182.901045436003,
  'input_type': 'unscaled'},
 'best_model': {'model_name': 'RandomForestRegressor',
  'best_r2_score': 0.4431143111886021,
  'input_type': 'unscaled'}}

In [12]:
train_scaled = pd.read_csv("artifacts/data_transformation/regression/train.csv")
train_unscaled = pd.read_csv("artifacts/data_transformation/ensemble/train.csv")

print(train_scaled.dtypes)
print("-" * 50)
print(train_unscaled.dtypes)


Product_id          float64
Product_Code        float64
Warehouse           float64
Product_Category    float64
Open                float64
Promo               float64
StateHoliday        float64
SchoolHoliday       float64
Petrol_price        float64
year                float64
month               float64
day                 float64
dayofweek           float64
weekofyear          float64
Order_Demand          int64
dtype: object
--------------------------------------------------
Product_id           int64
Product_Code         int64
Warehouse            int64
Product_Category     int64
Date                object
Order_Demand         int64
Open                 int64
Promo                int64
StateHoliday         int64
SchoolHoliday        int64
Petrol_price         int64
year                 int64
month                int64
day                  int64
dayofweek            int64
weekofyear           int64
dtype: object
